In [0]:
%skip
# Databricks notebook source
# 02_silver_transform_FIXED.py
# SOLUCIÓN: Limpieza robusta considerando tipos de datos
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# AJUSTE DE RUTAS A UNITY CATALOG VOLUMES
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
BRONZE_VOLUME_NAME = "bronce_data"
SILVER_VOLUME_NAME = "silver_data"

bronze_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{BRONZE_VOLUME_NAME}/"
silver_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{SILVER_VOLUME_NAME}/"

# *******************************************************************
# FORMATO DE FECHA/TIMESTAMP
# *******************************************************************
DATE_FORMAT = 'yyyy-MM-dd HH:mm:ss'
DATE_FORMAT_LITERAL = F.lit(DATE_FORMAT)

print("="*80)
print("🧹 TRANSFORMACIÓN SILVER - CON LIMPIEZA ROBUSTA V2")
print("="*80)

# *******************************************************************
# PASO 1: Crear el Volume de destino (Silver) si no existe
# *******************************************************************
try:
    print(f"\n📁 Verificando Volume Silver: {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    print("   ✅ Volume Silver verificado/creado")
except Exception as e:
    print(f"   ❌ ERROR al crear Volume Silver: {e}")
    raise

# *******************************************************************
# PASO 2: FUNCIÓN DE LIMPIEZA MEJORADA
# *******************************************************************
def clean_dataframe(df, df_name, numeric_cols=None, string_cols=None):
    """
    Limpia un DataFrame PySpark de forma inteligente según tipos de datos.
    
    Args:
        df: DataFrame de PySpark
        df_name: Nombre del DataFrame (para logging)
        numeric_cols: Lista de columnas que deben ser numéricas
        string_cols: Lista de columnas que deben ser strings
    
    Returns:
        DataFrame limpio
    """
    print(f"\n   🧹 Limpiando: {df_name}")
    initial_count = df.count()
    print(f"      Registros iniciales: {initial_count:,}")
    
    # ═══════════════════════════════════════════════════════════════
    # ESTRATEGIA: Convertir TODO a string primero, limpiar, luego tipar
    # ═══════════════════════════════════════════════════════════════
    
    # Paso 1: Convertir todas las columnas a string
    for col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(StringType()))
    
    # Paso 2: Reemplazar valores problemáticos (ahora son strings seguros)
    NULL_STRINGS = ['None', 'null', 'NULL', 'nan', 'NaN', 'none', 'NONE', '', 'NA', 'N/A']
    
    for col_name in df.columns:
        for null_str in NULL_STRINGS:
            df = df.withColumn(
                col_name,
                F.when(F.trim(F.col(col_name)) == null_str, None).otherwise(F.col(col_name))
            )
    
    # Paso 3: Convertir a tipos correctos
    if numeric_cols:
        for col_name in numeric_cols:
            if col_name in df.columns:
                # Usar try_cast para evitar errores (retorna null si falla)
                df = df.withColumn(
                    col_name,
                    F.expr(f"try_cast({col_name} as double)")
                )
                print(f"         ✓ {col_name} → DoubleType")
    
    if string_cols:
        for col_name in string_cols:
            if col_name in df.columns:
                # Ya son strings, solo asegurar que no sean null innecesarios
                df = df.withColumn(
                    col_name,
                    F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.col(col_name))
                )
                print(f"         ✓ {col_name} → StringType (limpio)")
    
    # Paso 4: Eliminar filas completamente vacías (todas las columnas null)
    df = df.dropna(how='all')
    
    # Paso 5: Eliminar duplicados
    df = df.dropDuplicates()
    
    final_count = df.count()
    removed = initial_count - final_count
    
    if removed > 0:
        print(f"      ⚠️  Registros removidos: {removed:,} (duplicados/vacíos)")
    
    print(f"      ✅ Registros finales: {final_count:,}")
    
    return df

# *******************************************************************
# PASO 3: Cargar y limpiar tablas Bronze
# *******************************************************************
print("\n📂 PROCESANDO TABLAS BRONZE\n")

# --- CUSTOMERS ---
print("👥 Procesando: CUSTOMERS")
customers = spark.read.parquet(bronze_path + "customers")
customers = clean_dataframe(
    customers,
    df_name="customers",
    string_cols=['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 
                 'customer_city', 'customer_state']
)

# --- ORDERS ---
print("\n📦 Procesando: ORDERS")
orders = spark.read.parquet(bronze_path + "orders")
orders = clean_dataframe(
    orders, 
    df_name="orders",
    string_cols=['order_id', 'customer_id', 'order_status']
)

# Convertir fechas (después de limpiar)
print("      📅 Convirtiendo timestamps...")
orders = orders \
    .withColumn("order_purchase_timestamp", 
                F.try_to_timestamp("order_purchase_timestamp", DATE_FORMAT_LITERAL)) \
    .withColumn("order_approved_at", 
                F.try_to_timestamp("order_approved_at", DATE_FORMAT_LITERAL)) \
    .withColumn("order_delivered_customer_date", 
                F.try_to_timestamp("order_delivered_customer_date", DATE_FORMAT_LITERAL)) \
    .withColumn("order_estimated_delivery_date", 
                F.try_to_timestamp("order_estimated_delivery_date", DATE_FORMAT_LITERAL))

# --- ORDER ITEMS ---
print("\n🛒 Procesando: ORDER_ITEMS")
order_items = spark.read.parquet(bronze_path + "order_items")
order_items = clean_dataframe(
    order_items,
    df_name="order_items",
    numeric_cols=['order_item_id', 'price', 'freight_value'],
    string_cols=['order_id', 'product_id', 'seller_id']
)

# Imputar valores numéricos con 0 (regla de negocio)
print("      🔧 Imputando valores nulos...")
order_items = order_items \
    .fillna(0.0, subset=['price', 'freight_value']) \
    .fillna(1, subset=['order_item_id'])

# --- ORDER PAYMENTS ---
print("\n💳 Procesando: ORDER_PAYMENTS")
order_payments = spark.read.parquet(bronze_path + "order_payments")
order_payments = clean_dataframe(
    order_payments,
    df_name="order_payments",
    numeric_cols=['payment_sequential', 'payment_installments', 'payment_value'],
    string_cols=['order_id', 'payment_type']
)

print("      🔧 Imputando valores nulos...")
order_payments = order_payments \
    .fillna(0.0, subset=['payment_value']) \
    .fillna(1, subset=['payment_installments', 'payment_sequential'])

# --- ORDER REVIEWS ---
print("\n⭐ Procesando: ORDER_REVIEWS")
order_reviews = spark.read.parquet(bronze_path + "order_reviews")
order_reviews = clean_dataframe(
    order_reviews,
    df_name="order_reviews",
    numeric_cols=['review_score'],
    string_cols=['review_id', 'order_id', 'review_comment_title', 'review_comment_message']
)

print("      📅 Convirtiendo timestamps de reviews...")
order_reviews = order_reviews \
    .withColumn("review_creation_date", 
                F.try_to_timestamp("review_creation_date", DATE_FORMAT_LITERAL)) \
    .withColumn("review_answer_timestamp", 
                F.try_to_timestamp("review_answer_timestamp", DATE_FORMAT_LITERAL))

print("      🔧 Imputando review_score nulo con 3 (neutro)...")
order_reviews = order_reviews.fillna(3.0, subset=['review_score'])

# --- PRODUCTS ---
print("\n📦 Procesando: PRODUCTS")
products = spark.read.parquet(bronze_path + "products")
products = clean_dataframe(
    products,
    df_name="products",
    numeric_cols=['product_name_lenght', 'product_description_lenght', 
                 'product_photos_qty', 'product_weight_g', 
                 'product_length_cm', 'product_height_cm', 'product_width_cm'],
    string_cols=['product_id', 'product_category_name']
)

print("      🔧 Imputando dimensiones con medianas...")
numeric_product_cols = ['product_name_lenght', 'product_description_lenght', 
                       'product_photos_qty', 'product_weight_g', 
                       'product_length_cm', 'product_height_cm', 'product_width_cm']

for col_name in numeric_product_cols:
    if col_name in products.columns:
        # Calcular mediana (aproximada)
        median_val = products.approxQuantile(col_name, [0.5], 0.01)
        if median_val and median_val[0] is not None:
            products = products.fillna(median_val[0], subset=[col_name])
            print(f"         ✓ {col_name}: mediana = {median_val[0]:.2f}")
        else:
            products = products.fillna(0.0, subset=[col_name])

# --- SELLERS ---
print("\n🏪 Procesando: SELLERS")
sellers = spark.read.parquet(bronze_path + "sellers")
sellers = clean_dataframe(
    sellers,
    df_name="sellers",
    string_cols=['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
)

# --- GEOLOCATION ---
print("\n🗺️  Procesando: GEOLOCATION")
geolocation = spark.read.parquet(bronze_path + "geolocation")
geolocation = clean_dataframe(
    geolocation,
    df_name="geolocation",
    numeric_cols=['geolocation_lat', 'geolocation_lng'],
    string_cols=['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']
)

print("      🔧 Filtrando coordenadas inválidas...")
# Filtrar coordenadas fuera de Brasil (lat: -34 a 5, lng: -74 a -34)
initial_geo = geolocation.count()
geolocation = geolocation.filter(
    F.col('geolocation_lat').isNotNull() &
    F.col('geolocation_lng').isNotNull() &
    F.col('geolocation_lat').between(-34, 5) &
    F.col('geolocation_lng').between(-74, -34)
)
final_geo = geolocation.count()
print(f"         ✓ Removidas: {initial_geo - final_geo:,} coordenadas inválidas")

# --- TRANSLATION ---
print("\n🌐 Procesando: TRANSLATION")
translation = spark.read.parquet(bronze_path + "translation")
translation = clean_dataframe(
    translation,
    df_name="translation",
    string_cols=['product_category_name', 'product_category_name_english']
)

# --- PREMIUM FLAG ---
print("\n⭐ Procesando: PREMIUM_FLAG")
premium_flag = spark.read.parquet(bronze_path + "premium_flag")

# Detectar columnas automáticamente
premium_cols = premium_flag.columns
numeric_premium = [col for col in premium_cols if 'premium' in col.lower() or 'flag' in col.lower()]
string_premium = [col for col in premium_cols if col not in numeric_premium]

premium_flag = clean_dataframe(
    premium_flag,
    df_name="premium_flag",
    numeric_cols=numeric_premium if numeric_premium else None,
    string_cols=string_premium if string_premium else None
)

# *******************************************************************
# PASO 4: AGREGACIONES
# *******************************************************************
print("\n" + "="*80)
print("🔄 GENERANDO AGREGACIONES")
print("="*80)

# Agregar order_items por order_id
print("\n📊 Agregando: order_items_agg")
order_items_agg = order_items.groupBy("order_id").agg(
    F.sum(F.coalesce(F.col("price"), F.lit(0))).alias("order_sum_price"),
    F.sum(F.coalesce(F.col("freight_value"), F.lit(0))).alias("order_sum_freight"),
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products")
)
print(f"   ✅ Generadas: {order_items_agg.count():,} agregaciones")

# Agregar payments por order_id
print("\n💰 Agregando: payments_agg")
payments_agg = order_payments.groupBy("order_id").agg(
    F.sum(F.coalesce(F.col("payment_value"), F.lit(0))).alias("payment_sum"),
    F.avg(F.coalesce(F.col("payment_installments"), F.lit(1))).alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print(f"   ✅ Generadas: {payments_agg.count():,} agregaciones")

# Join completo: orders + agregaciones + reviews
print("\n🔗 Uniendo: orders_full")
orders_full = orders \
    .join(order_items_agg, on="order_id", how="left") \
    .join(payments_agg, on="order_id", how="left") \
    .join(
        order_reviews.select("order_id", "review_score", "review_comment_message", "review_creation_date"),
        on="order_id",
        how="left"
    )

# Imputar nulls en agregaciones (órdenes sin items = 0)
print("   🔧 Imputando valores nulos en agregaciones...")
orders_full = orders_full \
    .fillna(0.0, subset=['order_sum_price', 'order_sum_freight', 'items_count', 
                        'distinct_products', 'payment_sum', 'n_payment_types']) \
    .fillna(1.0, subset=['avg_installments'])

print(f"   ✅ orders_full: {orders_full.count():,} registros")

# *******************************************************************
# PASO 5: VALIDACIÓN DE CALIDAD
# *******************************************************************
print("\n" + "="*80)
print("✅ VALIDACIÓN DE CALIDAD DE DATOS")
print("="*80)

def validate_dataframe(df, df_name, sample_size=5):
    """Valida calidad de datos y muestra estadísticas"""
    print(f"\n🔍 Validando: {df_name}")
    
    total_rows = df.count()
    print(f"   Total registros: {total_rows:,}")
    print(f"   Columnas: {len(df.columns)}")
    
    # Contar nulos por columna (top 5)
    null_counts = []
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            pct = (null_count / total_rows) * 100
            null_counts.append((col_name, null_count, pct))
    
    if null_counts:
        print(f"\n   ⚠️  Columnas con nulos (top {sample_size}):")
        for col, count, pct in sorted(null_counts, key=lambda x: x[2], reverse=True)[:sample_size]:
            print(f"      - {col}: {count:,} ({pct:.1f}%)")
    else:
        print(f"   ✅ Sin valores nulos")
    
    return df

# Validar tablas principales
customers = validate_dataframe(customers, "customers")
orders_full = validate_dataframe(orders_full, "orders_full")
products = validate_dataframe(products, "products")
sellers = validate_dataframe(sellers, "sellers")

# *******************************************************************
# PASO 6: GUARDAR EN SILVER
# *******************************************************************
print("\n" + "="*80)
print("💾 GUARDANDO EN SILVER LAYER")
print("="*80)

tables_to_save = {
    "customers": customers,
    "orders_full": orders_full,
    "order_items": order_items,
    "order_payments_agg": payments_agg,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation,
    "premium_flag": premium_flag
}

for table_name, df in tables_to_save.items():
    output_path = silver_path + table_name
    record_count = df.count()
    
    print(f"\n💾 Guardando: {table_name}")
    print(f"   Registros: {record_count:,}")
    print(f"   Columnas: {len(df.columns)}")
    
    df.write.mode("overwrite").parquet(output_path)
    
    print(f"   ✅ Guardado en: {output_path}")

# *******************************************************************
# RESUMEN FINAL
# *******************************************************************
print("\n" + "="*80)
print("✅ TRANSFORMACIÓN SILVER COMPLETADA EXITOSAMENTE")
print("="*80)
print(f"\n📁 Ubicación: {silver_path}")
print(f"📊 Tablas generadas: {len(tables_to_save)}")
print(f"\n📋 Resumen de tablas:")
for name in tables_to_save.keys():
    print(f"   ✓ {name}")
print("\n🎯 Próximo paso: Ejecutar 03_gold_features.py")
print("="*80)

In [0]:
# Databricks notebook source
# 02_silver_transform_SERVERLESS.py
# OPTIMIZADO PARA DATABRICKS SERVERLESS (sin cache)
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, StringType
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# CONFIGURACIÓN
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
BRONZE_VOLUME_NAME = "bronce_data"
SILVER_VOLUME_NAME = "silver_data"

bronze_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{BRONZE_VOLUME_NAME}/"
silver_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{SILVER_VOLUME_NAME}/"

DATE_FORMAT = 'yyyy-MM-dd HH:mm:ss'
DATE_FORMAT_LITERAL = F.lit(DATE_FORMAT)

print("="*80)
print("🚀 TRANSFORMACIÓN SILVER - SERVERLESS")
print("="*80)

# *******************************************************************
# PASO 1: Crear Volume Silver
# *******************************************************************
try:
    print(f"\n📁 Verificando Volume Silver...")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    print("   ✅ Volume verificado")
except Exception as e:
    print(f"   ❌ ERROR: {e}")
    raise

# *******************************************************************
# PASO 2: FUNCIÓN DE LIMPIEZA SIMPLE Y RÁPIDA
# *******************************************************************
def clean_dataframe(df, numeric_cols=None, string_cols=None):
    """
    Limpieza rápida sin cache (compatible con Serverless)
    """
    # 1. Convertir a string
    for col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(StringType()))
    
    # 2. Limpiar valores nulos (una sola expresión)
    null_values = ['None', 'null', 'NULL', 'nan', 'NaN', 'none', 'NONE', '', 'NA']
    
    for col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)).isin(null_values), None)
             .otherwise(F.col(col_name))
        )
    
    # 3. Convertir tipos
    if numeric_cols:
        for col_name in numeric_cols:
            if col_name in df.columns:
                df = df.withColumn(col_name, F.expr(f"try_cast({col_name} as double)"))
    
    if string_cols:
        for col_name in string_cols:
            if col_name in df.columns:
                df = df.withColumn(col_name, F.trim(F.col(col_name)))
    
    # 4. Limpiar duplicados
    df = df.dropna(how='all').dropDuplicates()
    
    return df

# *******************************************************************
# PASO 3: PROCESAR TABLAS
# *******************************************************************
print("\n📂 PROCESANDO TABLAS...\n")

# --- CUSTOMERS ---
print("👥 customers...", end=" ")
customers = spark.read.parquet(bronze_path + "customers")
customers = clean_dataframe(
    customers,
    string_cols=['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 
                 'customer_city', 'customer_state']
)
print("✅")

# --- ORDERS ---
print("📦 orders...", end=" ")
orders = spark.read.parquet(bronze_path + "orders")
orders = clean_dataframe(
    orders,
    string_cols=['order_id', 'customer_id', 'order_status']
)

# Convertir fechas
orders = orders \
    .withColumn("order_purchase_timestamp", F.try_to_timestamp("order_purchase_timestamp", DATE_FORMAT_LITERAL)) \
    .withColumn("order_approved_at", F.try_to_timestamp("order_approved_at", DATE_FORMAT_LITERAL)) \
    .withColumn("order_delivered_customer_date", F.try_to_timestamp("order_delivered_customer_date", DATE_FORMAT_LITERAL)) \
    .withColumn("order_estimated_delivery_date", F.try_to_timestamp("order_estimated_delivery_date", DATE_FORMAT_LITERAL))
print("✅")

# --- ORDER ITEMS ---
print("🛒 order_items...", end=" ")
order_items = spark.read.parquet(bronze_path + "order_items")
order_items = clean_dataframe(
    order_items,
    numeric_cols=['order_item_id', 'price', 'freight_value'],
    string_cols=['order_id', 'product_id', 'seller_id']
)
order_items = order_items.fillna(0.0, subset=['price', 'freight_value', 'order_item_id'])
print("✅")

# --- ORDER PAYMENTS ---
print("💳 order_payments...", end=" ")
order_payments = spark.read.parquet(bronze_path + "order_payments")
order_payments = clean_dataframe(
    order_payments,
    numeric_cols=['payment_sequential', 'payment_installments', 'payment_value'],
    string_cols=['order_id', 'payment_type']
)
order_payments = order_payments.fillna(0.0, subset=['payment_value']).fillna(1, subset=['payment_installments', 'payment_sequential'])
print("✅")

# --- ORDER REVIEWS ---
print("⭐ order_reviews...", end=" ")
order_reviews = spark.read.parquet(bronze_path + "order_reviews")
order_reviews = clean_dataframe(
    order_reviews,
    numeric_cols=['review_score'],
    string_cols=['review_id', 'order_id', 'review_comment_title', 'review_comment_message']
)
order_reviews = order_reviews \
    .withColumn("review_creation_date", F.try_to_timestamp("review_creation_date", DATE_FORMAT_LITERAL)) \
    .withColumn("review_answer_timestamp", F.try_to_timestamp("review_answer_timestamp", DATE_FORMAT_LITERAL)) \
    .fillna(3.0, subset=['review_score'])
print("✅")

# --- PRODUCTS ---
print("📦 products...", end=" ")
products = spark.read.parquet(bronze_path + "products")
products = clean_dataframe(
    products,
    numeric_cols=['product_name_lenght', 'product_description_lenght', 
                 'product_photos_qty', 'product_weight_g', 
                 'product_length_cm', 'product_height_cm', 'product_width_cm'],
    string_cols=['product_id', 'product_category_name']
)
# Imputar con 0 (más rápido que mediana)
products = products.fillna(0.0, subset=['product_name_lenght', 'product_description_lenght', 
                                       'product_photos_qty', 'product_weight_g', 
                                       'product_length_cm', 'product_height_cm', 'product_width_cm'])
print("✅")

# --- SELLERS ---
print("🏪 sellers...", end=" ")
sellers = spark.read.parquet(bronze_path + "sellers")
sellers = clean_dataframe(
    sellers,
    string_cols=['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
)
print("✅")

# --- GEOLOCATION ---
print("🗺️  geolocation...", end=" ")
geolocation = spark.read.parquet(bronze_path + "geolocation")
geolocation = clean_dataframe(
    geolocation,
    numeric_cols=['geolocation_lat', 'geolocation_lng'],
    string_cols=['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']
)
# Filtrar coordenadas inválidas
geolocation = geolocation.filter(
    F.col('geolocation_lat').isNotNull() &
    F.col('geolocation_lng').isNotNull() &
    F.col('geolocation_lat').between(-34, 5) &
    F.col('geolocation_lng').between(-74, -34)
)
print("✅")

# --- TRANSLATION ---
print("🌐 translation...", end=" ")
translation = spark.read.parquet(bronze_path + "translation")
translation = clean_dataframe(
    translation,
    string_cols=['product_category_name', 'product_category_name_english']
)
print("✅")

# --- PREMIUM FLAG ---
print("⭐ premium_flag...", end=" ")
premium_flag = spark.read.parquet(bronze_path + "premium_flag")
premium_cols = premium_flag.columns
numeric_premium = [col for col in premium_cols if 'premium' in col.lower() or 'flag' in col.lower()]
string_premium = [col for col in premium_cols if col not in numeric_premium]
premium_flag = clean_dataframe(
    premium_flag,
    numeric_cols=numeric_premium if numeric_premium else None,
    string_cols=string_premium if string_premium else None
)
print("✅")

# *******************************************************************
# PASO 4: AGREGACIONES
# *******************************************************************
print("\n" + "="*80)
print("🔄 AGREGACIONES")
print("="*80)

print("\n📊 order_items_agg...", end=" ")
order_items_agg = order_items.groupBy("order_id").agg(
    F.sum(F.coalesce(F.col("price"), F.lit(0))).alias("order_sum_price"),
    F.sum(F.coalesce(F.col("freight_value"), F.lit(0))).alias("order_sum_freight"),
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products")
)
print("✅")

print("💰 payments_agg...", end=" ")
payments_agg = order_payments.groupBy("order_id").agg(
    F.sum(F.coalesce(F.col("payment_value"), F.lit(0))).alias("payment_sum"),
    F.avg(F.coalesce(F.col("payment_installments"), F.lit(1))).alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)
print("✅")

print("🔗 orders_full...", end=" ")
orders_full = orders \
    .join(order_items_agg, on="order_id", how="left") \
    .join(payments_agg, on="order_id", how="left") \
    .join(
        order_reviews.select("order_id", "review_score", "review_comment_message", "review_creation_date"),
        on="order_id",
        how="left"
    )

# Imputar nulls en agregaciones
orders_full = orders_full \
    .fillna(0.0, subset=['order_sum_price', 'order_sum_freight', 'items_count', 
                        'distinct_products', 'payment_sum', 'n_payment_types']) \
    .fillna(1.0, subset=['avg_installments'])
print("✅")

# *******************************************************************
# PASO 5: GUARDAR EN SILVER (DIRECTAMENTE, SIN VALIDACIONES COSTOSAS)
# *******************************************************************
print("\n" + "="*80)
print("💾 GUARDANDO EN SILVER")
print("="*80)

tables_to_save = {
    "customers": customers,
    "orders_full": orders_full,
    "order_items": order_items,
    "order_payments_agg": payments_agg,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation,
    "premium_flag": premium_flag
}

print()
for table_name, df in tables_to_save.items():
    output_path = silver_path + table_name
    print(f"💾 {table_name:25s}...", end=" ")
    
    # Guardar directamente (Spark optimiza automáticamente en Serverless)
    df.write.mode("overwrite").parquet(output_path)
    
    print("✅")

# *******************************************************************
# RESUMEN FINAL
# *******************************************************************
print("\n" + "="*80)
print("✅ TRANSFORMACIÓN COMPLETADA")
print("="*80)
print(f"\n📁 Silver Layer: {silver_path}")
print(f"📊 Tablas generadas: {len(tables_to_save)}")
print("\n📋 Lista de tablas:")
for name in tables_to_save.keys():
    print(f"   ✓ {name}")
print("\n🎯 Siguiente paso: Ejecutar 03_gold_features.py")
print("="*80)

In [0]:
%skip
# 02_silver_transform.py
# Notebook: 02_silver_transform
# Ejecutar en Databricks (Python)

from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# AJUSTE DE RUTAS A UNITY CATALOG VOLUMES
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
BRONZE_VOLUME_NAME = "bronce_data"
SILVER_VOLUME_NAME = "silver_data"

# Rutas de Origen y Destino
bronze_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{BRONZE_VOLUME_NAME}/"
silver_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{SILVER_VOLUME_NAME}/"

# *******************************************************************
# PASO 1: Crear el Volume de destino (Silver) si no existe
# *******************************************************************
try:
    print(f"Verificando y creando el Volume de destino Silver: {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{SILVER_VOLUME_NAME}")
    print("Volume Silver verificado/creado exitosamente.")
except Exception as e:
    print(f"ERROR: No se pudo crear el Volume Silver. Verifica tus permisos de Unity Catalog.")
    print(f"Detalle del error de creación: {e}")
    raise # Detener la ejecución si el destino no se puede asegurar

# *******************************************************************
# PASO 2: Cargar todas las tablas bronze (incluyendo las nuevas)
# *******************************************************************
print("Cargando tablas de la capa Bronze...")
customers = spark.read.parquet(bronze_path + "customers")
orders = spark.read.parquet(bronze_path + "orders")
order_items = spark.read.parquet(bronze_path + "order_items")
order_payments = spark.read.parquet(bronze_path + "order_payments")
order_reviews = spark.read.parquet(bronze_path + "order_reviews")
products = spark.read.parquet(bronze_path + "products")
sellers = spark.read.parquet(bronze_path + "sellers")

# Archivos adicionales
geolocation = spark.read.parquet(bronze_path + "geolocation")
translation = spark.read.parquet(bronze_path + "translation")
premium_flag = spark.read.parquet(bronze_path + "premium_flag")


# *******************************************************************
# PASO 3: Transformaciones (Convertir tipos de datos y agregaciones)
# *******************************************************************

# CORRECCIÓN DE ERROR [CANNOT_PARSE_TIMESTAMP]: Se usa F.try_to_timestamp.
# CORRECCIÓN DE ERROR [UNRESOLVED_COLUMN]: Se envuelve la variable DATE_FORMAT en F.lit().
DATE_FORMAT = 'yyyy-MM-dd HH:mm:ss'
DATE_FORMAT_LITERAL = F.lit(DATE_FORMAT) # Definimos el literal de Spark una vez

print("Realizando transformaciones de fechas y agregaciones...")
orders = orders \
    .withColumn("order_purchase_timestamp", F.try_to_timestamp("order_purchase_timestamp", DATE_FORMAT_LITERAL)) \
    .withColumn("order_approved_at", F.try_to_timestamp("order_approved_at", DATE_FORMAT_LITERAL)) \
    .withColumn("order_delivered_customer_date", F.try_to_timestamp("order_delivered_customer_date", DATE_FORMAT_LITERAL)) \
    .withColumn("order_estimated_delivery_date", F.try_to_timestamp("order_estimated_delivery_date", DATE_FORMAT_LITERAL))

# Aplicar formato explícito y try_to_timestamp a order_reviews
order_reviews = order_reviews \
    .withColumn("review_creation_date", F.try_to_timestamp("review_creation_date", DATE_FORMAT_LITERAL)) \
    .withColumn("review_answer_timestamp", F.try_to_timestamp("review_answer_timestamp", DATE_FORMAT_LITERAL))

# Agregar resumen por order_id (suma de items, conteos)
order_items_agg = order_items.groupBy("order_id").agg(
    F.sum(F.col("price")).alias("order_sum_price"),
    F.sum(F.col("freight_value")).alias("order_sum_freight"),
    F.count("*").alias("items_count"),
    F.countDistinct("product_id").alias("distinct_products")
)

# Agregar pagos por order
payments_agg = order_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("payment_sum"),
    F.avg("payment_installments").alias("avg_installments"),
    F.countDistinct("payment_type").alias("n_payment_types")
)

# Join: orders + items + payments + reviews (left joins para no perder orders)
orders_full = orders.join(order_items_agg, on="order_id", how="left") \
                    .join(payments_agg, on="order_id", how="left") \
                    .join(order_reviews.select("order_id", "review_score", "review_comment_message", "review_creation_date"), on="order_id", how="left")

# *******************************************************************
# PASO 4: Guardar Silver (Incluyendo las nuevas tablas no transformadas)
# *******************************************************************
print("Guardando tablas en la capa Silver...")
customers.write.mode("overwrite").parquet(silver_path + "customers")
orders_full.write.mode("overwrite").parquet(silver_path + "orders_full")
order_items.write.mode("overwrite").parquet(silver_path + "order_items")
payments_agg.write.mode("overwrite").parquet(silver_path + "order_payments_agg")
products.write.mode("overwrite").parquet(silver_path + "products")
sellers.write.mode("overwrite").parquet(silver_path + "sellers")

# Guardar los archivos adicionales en Silver, tal cual (sin transformaciones en esta etapa)
geolocation.write.mode("overwrite").parquet(silver_path + "geolocation")
translation.write.mode("overwrite").parquet(silver_path + "translation")
premium_flag.write.mode("overwrite").parquet(silver_path + "premium_flag")


print("Transformación Silver completada.")
